# 3-Stock Quantum Objective Variable Demonstration

This notebook demonstrates how the quantum optimization objective variable is constructed for portfolio optimization using a simple 3-stock toy model.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from dimod import Integer, ConstrainedQuadraticModel, quicksum
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("📊 3-Stock Quantum Portfolio Objective Variable Demo")
print("=" * 60)

📊 3-Stock Quantum Portfolio Objective Variable Demo


In [2]:
# Create toy data for 3 stocks over 30 days
dates = pd.date_range('2024-01-01', periods=30, freq='D')
stocks = ['AAPL', 'GOOGL', 'MSFT']

# Generate synthetic price data
prices_data = {}
for stock in stocks:
    # Start with base price and add random walk
    base_price = np.random.uniform(100, 200)
    returns = np.random.normal(0.001, 0.02, 30)  # Daily returns ~0.1% mean, 2% std
    prices = [base_price]
    for ret in returns[1:]:
        prices.append(prices[-1] * (1 + ret))
    prices_data[stock] = prices

toy_data = pd.DataFrame(prices_data, index=dates)
print("3-Stock Toy Model Data (first 10 days):")
print(toy_data.head(10).round(2))

# Calculate returns and covariance matrix
returns = np.log(toy_data) - np.log(toy_data.shift(1))
returns = returns.dropna()

print(f"\nReturns data shape: {returns.shape}")
print("\nAverage daily returns:")
avg_returns = returns.mean()
for stock, ret in avg_returns.items():
    print(f"  {stock}: {ret:.6f} ({ret*252*100:.2f}% annualized)")

print(f"\nCovariance matrix:")
cov_matrix = returns.cov()
print(cov_matrix.round(8))

3-Stock Toy Model Data (first 10 days):
              AAPL   GOOGL    MSFT
2024-01-01  137.45  190.93  172.96
2024-01-02  138.47  186.46  174.16
2024-01-03  139.38  187.43  170.72
2024-01-04  142.34  180.27  170.63
2024-01-05  140.82  175.66  174.12
2024-01-06  139.49  176.53  177.07
2024-01-07  138.03  179.31  182.54
2024-01-08  135.62  180.10  183.96
2024-01-09  128.67  179.87  196.55
2024-01-10  131.24  178.96  193.13

Returns data shape: (29, 3)

Average daily returns:
  AAPL: -0.003736 (-94.14% annualized)
  GOOGL: -0.004399 (-110.86% annualized)
  MSFT: 0.005452 (137.39% annualized)

Covariance matrix:
           AAPL     GOOGL      MSFT
AAPL   0.000415 -0.000035 -0.000203
GOOGL -0.000035  0.000302  0.000045
MSFT  -0.000203  0.000045  0.000390


In [3]:
# Setup optimization parameters
print("=" * 60)
print("QUANTUM OPTIMIZATION SETUP")
print("=" * 60)

# Parameters for optimization
budget = 1000
min_weight = 0.05  # 5% minimum per stock
risk_aversion = 0.5

# Current prices (last day)
prices = toy_data.iloc[-1, :]
print(f"Current prices: {dict(prices.round(2))}")

# Calculate bounds for decision variables
min_budget_per_stock = budget * min_weight
min_shares = (min_budget_per_stock / prices).astype(int).clip(lower=1)
max_shares = (budget / prices * 0.4).astype(int)  # Max 40% per stock

print(f"\nDecision variable bounds:")
for stock in stocks:
    print(f"  x_{stock}: [{min_shares[stock]}, {max_shares[stock]}] shares")
    print(f"    Min investment: ${min_shares[stock] * prices[stock]:.2f}")
    print(f"    Max investment: ${max_shares[stock] * prices[stock]:.2f}")

# Create the CQM model
cqm = ConstrainedQuadraticModel()

# Decision variables: number of shares to buy
x = {s: Integer(f"x_{s}", 
               lower_bound=min_shares[s], 
               upper_bound=max_shares[s]) for s in stocks}

print(f"\nDecision variables created:")
for stock, var in x.items():
    print(f"  {var}")

QUANTUM OPTIMIZATION SETUP
Current prices: {'AAPL': np.float64(123.34), 'GOOGL': np.float64(168.06), 'MSFT': np.float64(202.59)}

Decision variable bounds:
  x_AAPL: [1, 3] shares
    Min investment: $123.34
    Max investment: $370.03
  x_GOOGL: [1, 2] shares
    Min investment: $168.06
    Max investment: $336.13
  x_MSFT: [1, 1] shares
    Min investment: $202.59
    Max investment: $202.59

Decision variables created:
  QuadraticModel({'x_AAPL': 1.0}, {}, 0.0, {'x_AAPL': 'INTEGER'}, dtype='float64')
  QuadraticModel({'x_GOOGL': 1.0}, {}, 0.0, {'x_GOOGL': 'INTEGER'}, dtype='float64')
  QuadraticModel({'x_MSFT': 1.0}, {}, 0.0, {'x_MSFT': 'INTEGER'}, dtype='float64')


In [4]:
# Build the objective function components
print("\n" + "=" * 60)
print("OBJECTIVE FUNCTION COMPONENTS")
print("=" * 60)

# 1. RISK COMPONENT (Portfolio Variance)
print("\n1. RISK COMPONENT (Quadratic Terms)")
print("-" * 40)

risk_component = 0
risk_terms = []

print("Risk = Σᵢⱼ σᵢⱼ × (xᵢ × priceᵢ) × (xⱼ × priceⱼ)")
print("Where σᵢⱼ is the covariance between stocks i and j\n")

for i, s1 in enumerate(stocks):
    for j, s2 in enumerate(stocks):
        coeff = cov_matrix.iloc[i, j] * prices[s1] * prices[s2]
        if abs(coeff) > 1e-10:  # Only show significant terms
            risk_terms.append((s1, s2, coeff))
            risk_component += coeff * x[s1] * x[s2]
            
            if i == j:
                print(f"  Variance term: σ_{s1},{s1} × price_{s1}² × x_{s1}²")
                print(f"    = {cov_matrix.iloc[i, j]:.8f} × {prices[s1]:.2f}² × x_{s1}²")
                print(f"    = {coeff:.8f} × x_{s1}²")
            else:
                print(f"  Covariance term: σ_{s1},{s2} × price_{s1} × price_{s2} × x_{s1} × x_{s2}")
                print(f"    = {cov_matrix.iloc[i, j]:.8f} × {prices[s1]:.2f} × {prices[s2]:.2f} × x_{s1} × x_{s2}")
                print(f"    = {coeff:.8f} × x_{s1} × x_{s2}")

print(f"\nTotal risk terms: {len(risk_terms)}")


OBJECTIVE FUNCTION COMPONENTS

1. RISK COMPONENT (Quadratic Terms)
----------------------------------------
Risk = Σᵢⱼ σᵢⱼ × (xᵢ × priceᵢ) × (xⱼ × priceⱼ)
Where σᵢⱼ is the covariance between stocks i and j

  Variance term: σ_AAPL,AAPL × price_AAPL² × x_AAPL²
    = 0.00041455 × 123.34² × x_AAPL²
    = 6.30661125 × x_AAPL²
  Covariance term: σ_AAPL,GOOGL × price_AAPL × price_GOOGL × x_AAPL × x_GOOGL
    = -0.00003531 × 123.34 × 168.06 × x_AAPL × x_GOOGL
    = -0.73199548 × x_AAPL × x_GOOGL
  Covariance term: σ_AAPL,MSFT × price_AAPL × price_MSFT × x_AAPL × x_MSFT
    = -0.00020316 × 123.34 × 202.59 × x_AAPL × x_MSFT
    = -5.07647254 × x_AAPL × x_MSFT
  Covariance term: σ_GOOGL,AAPL × price_GOOGL × price_AAPL × x_GOOGL × x_AAPL
    = -0.00003531 × 168.06 × 123.34 × x_GOOGL × x_AAPL
    = -0.73199548 × x_GOOGL × x_AAPL
  Variance term: σ_GOOGL,GOOGL × price_GOOGL² × x_GOOGL²
    = 0.00030250 × 168.06² × x_GOOGL²
    = 8.54410470 × x_GOOGL²
  Covariance term: σ_GOOGL,MSFT × price_GOOGL ×

In [5]:
# 2. RETURN COMPONENT (Linear Terms)
print("\n2. RETURN COMPONENT (Linear Terms)")
print("-" * 40)

return_component = 0
return_terms = []

print("Expected Return = Σᵢ μᵢ × (xᵢ × priceᵢ)")
print("Where μᵢ is the expected return of stock i\n")

for i, stock in enumerate(stocks):
    coeff = avg_returns.iloc[i] * prices[stock]
    return_terms.append((stock, coeff))
    return_component += coeff * x[stock]
    
    print(f"  Return term: μ_{stock} × price_{stock} × x_{stock}")
    print(f"    = {avg_returns.iloc[i]:.8f} × {prices[stock]:.2f} × x_{stock}")
    print(f"    = {coeff:.8f} × x_{stock}")

print(f"\nAll return terms:")
for stock, coeff in return_terms:
    print(f"  {coeff:.8f} × x_{stock}")


2. RETURN COMPONENT (Linear Terms)
----------------------------------------
Expected Return = Σᵢ μᵢ × (xᵢ × priceᵢ)
Where μᵢ is the expected return of stock i

  Return term: μ_AAPL × price_AAPL × x_AAPL
    = -0.00373556 × 123.34 × x_AAPL
    = -0.46074961 × x_AAPL
  Return term: μ_GOOGL × price_GOOGL × x_GOOGL
    = -0.00439913 × 168.06 × x_GOOGL
    = -0.73933389 × x_GOOGL
  Return term: μ_MSFT × price_MSFT × x_MSFT
    = 0.00545180 × 202.59 × x_MSFT
    = 1.10446093 × x_MSFT

All return terms:
  -0.46074961 × x_AAPL
  -0.73933389 × x_GOOGL
  1.10446093 × x_MSFT


In [6]:
# 3. COMBINED OBJECTIVE FUNCTION
print("\n3. COMBINED OBJECTIVE FUNCTION")
print("-" * 40)

# Scale return component to balance with risk
scale_factor = 1.0 / risk_aversion
print(f"Risk aversion parameter: {risk_aversion}")
print(f"Return scale factor: {scale_factor}")

# Combined objective: Minimize risk + Minimize (-returns * scale_factor)
# This effectively maximizes the Sharpe ratio
objective = risk_component - scale_factor * return_component

print(f"\nObjective = Risk - (Scale Factor × Returns)")
print(f"Objective = Risk - ({scale_factor} × Returns)")

print(f"\nThis means we want to:")
print(f"  • MINIMIZE portfolio risk (variance)")
print(f"  • MAXIMIZE expected returns (by minimizing negative returns)")
print(f"  • Balance controlled by risk_aversion = {risk_aversion}")

# Set the objective in the CQM
cqm.set_objective(objective)

print(f"\n✅ Objective function set in CQM model")


3. COMBINED OBJECTIVE FUNCTION
----------------------------------------
Risk aversion parameter: 0.5
Return scale factor: 2.0

Objective = Risk - (Scale Factor × Returns)
Objective = Risk - (2.0 × Returns)

This means we want to:
  • MINIMIZE portfolio risk (variance)
  • MAXIMIZE expected returns (by minimizing negative returns)
  • Balance controlled by risk_aversion = 0.5

✅ Objective function set in CQM model


In [7]:
# 4. CONSTRAINTS
print("\n4. CONSTRAINTS")
print("-" * 40)

# Budget constraint (invest close to full budget)
budget_expr = quicksum([x[s] * prices[s] for s in stocks])
cqm.add_constraint(budget_expr <= budget, label='budget_upper')
cqm.add_constraint(budget_expr >= budget * 0.9, label='budget_lower')

print(f"Budget constraints:")
print(f"  {budget * 0.9:.2f} ≤ Σ(xᵢ × priceᵢ) ≤ {budget:.2f}")
print(f"  {budget * 0.9:.2f} ≤ ({prices['AAPL']:.2f}×x_AAPL + {prices['GOOGL']:.2f}×x_GOOGL + {prices['MSFT']:.2f}×x_MSFT) ≤ {budget:.2f}")

# Minimum weight constraints (already built into variable bounds)
print(f"\nMinimum weight constraints (via variable bounds):")
for s in stocks:
    print(f"  x_{s} ≥ {min_shares[s]} shares (ensures ≥{min_weight*100}% allocation)")

print(f"\n✅ All constraints added to CQM model")


4. CONSTRAINTS
----------------------------------------
Budget constraints:
  900.00 ≤ Σ(xᵢ × priceᵢ) ≤ 1000.00
  900.00 ≤ (123.34×x_AAPL + 168.06×x_GOOGL + 202.59×x_MSFT) ≤ 1000.00

Minimum weight constraints (via variable bounds):
  x_AAPL ≥ 1 shares (ensures ≥5.0% allocation)
  x_GOOGL ≥ 1 shares (ensures ≥5.0% allocation)
  x_MSFT ≥ 1 shares (ensures ≥5.0% allocation)

✅ All constraints added to CQM model


In [8]:
# 5. FINAL MODEL SUMMARY
print("\n" + "=" * 60)
print("FINAL QUANTUM MODEL SUMMARY")
print("=" * 60)

print(f"Model type: Constrained Quadratic Model (CQM)")
print(f"Decision variables: {len(x)} integer variables (number of shares)")
print(f"Objective terms: {len(risk_terms)} quadratic + {len(return_terms)} linear")
print(f"Constraints: {len(cqm.constraints)} constraints")

print(f"\nDecision Variables:")
for stock in stocks:
    print(f"  x_{stock}: integer in [{min_shares[stock]}, {max_shares[stock]}]")

print(f"\nObjective Function Structure:")
print(f"  • Quadratic terms (risk): {len([t for t in risk_terms if t[0] == t[1]])} diagonal + {len([t for t in risk_terms if t[0] != t[1]])} off-diagonal")
print(f"  • Linear terms (returns): {len(return_terms)}")
print(f"  • Goal: Maximize Sharpe ratio ≈ Returns/Risk")

print(f"\nConstraints:")
for label in cqm.constraints:
    print(f"  • {label}")

print(f"\n🎯 This model can be solved using:")
print(f"   • LeapHybridCQMSampler() - D-Wave quantum cloud")
print(f"   • Classical solvers for comparison")


FINAL QUANTUM MODEL SUMMARY
Model type: Constrained Quadratic Model (CQM)
Decision variables: 3 integer variables (number of shares)
Objective terms: 9 quadratic + 3 linear
Constraints: 2 constraints

Decision Variables:
  x_AAPL: integer in [1, 3]
  x_GOOGL: integer in [1, 2]
  x_MSFT: integer in [1, 1]

Objective Function Structure:
  • Quadratic terms (risk): 3 diagonal + 6 off-diagonal
  • Linear terms (returns): 3
  • Goal: Maximize Sharpe ratio ≈ Returns/Risk

Constraints:
  • budget_upper
  • budget_lower

🎯 This model can be solved using:
   • LeapHybridCQMSampler() - D-Wave quantum cloud
   • Classical solvers for comparison
